<a href="https://colab.research.google.com/github/empress20/African_Recipes/blob/main/QLoRA1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Clean African Recipe QLoRA Training with Complete Evaluation Metrics
# Addresses dataset issues and includes all methodology metrics

# ================================
# CELL 1: Install Dependencies
# ================================

!pip install -q unsloth datasets accelerate bitsandbytes trl transformers peft pandas openpyxl rouge-score nltk textstat
!python -c "import nltk; nltk.download('punkt'); nltk.download('stopwords'); nltk.download('wordnet')"

# ================================
# CELL 2: Upload and Clean Dataset
# ================================

from google.colab import files
import pandas as pd
import re
from html import unescape
import unicodedata

# Upload dataset
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

def clean_dataset(df):
    def clean_text(text):
        if pd.isna(text):
            return ""
        text = str(text)
        text = unescape(text)
        text = unicodedata.normalize('NFKD', text)
        text = text.replace('½', '1/2').replace('¼', '1/4').replace('¾', '3/4')
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    # Clean all columns
    for col in df.columns:
        df[col] = df[col].apply(clean_text)

    # Remove invalid recipes
    df = df.dropna(subset=['Dish-Name', 'Instructions'])
    df = df[df['Instructions'].str.len() > 50]
    df = df[df['Dish-Name'].str.len() > 2]

    # Fix instructions formatting
    def clean_instructions(instructions):
        instructions = re.sub(r'STEP\s*(\d+)', r'Step \1:', instructions)
        instructions = re.sub(r'Step\s*(\d+)(?!:)', r'Step \1:', instructions)

        # Split long sentences
        sentences = instructions.split('.')
        cleaned_sentences = []
        for sentence in sentences:
            sentence = sentence.strip()
            if len(sentence) > 150:
                parts = re.split(r'\s+(and then|then|meanwhile|after|next)\s+', sentence, flags=re.IGNORECASE)
                for part in parts:
                    if len(part.strip()) > 10:
                        cleaned_sentences.append(part.strip())
            elif len(sentence) > 10:
                cleaned_sentences.append(sentence)
        return '. '.join(cleaned_sentences) + '.'

    df['Instructions'] = df['Instructions'].apply(clean_instructions)

    # Format for training
    formatted_data = []
    for _, row in df.iterrows():
        instruction = f"Generate a complete recipe for {row['Dish-Name']}."
        if 'About' in row and row['About']:
            instruction += f"\nAbout: {row['About'][:200]}"
        if 'Ingredients' in row and row['Ingredients']:
            instruction += f"\nIngredients: {row['Ingredients'][:300]}"

        response = f"# {row['Dish-Name']}\n\n**Instructions:**\n{row['Instructions']}"

        formatted_data.append({
            'text': f"### Instruction:\n{instruction}\n\n### Response:\n{response}"
        })

    return pd.DataFrame(formatted_data)

cleaned_df = clean_dataset(df)
print(f"Processed {len(cleaned_df)} recipes")

# ================================
# CELL 3: Setup Model
# ================================

import torch
from unsloth import FastLanguageModel
from datasets import Dataset

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-2-2b-it-bnb-4bit",
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

FastLanguageModel.for_training(model)
model.print_trainable_parameters()

# ================================
# CELL 4: Train Model with Logging
# ================================

from trl import SFTTrainer, SFTConfig
import matplotlib.pyplot as plt

train_dataset = Dataset.from_pandas(cleaned_df)
from transformers.trainer_callback import TrainerCallback

class MetricsCallback(TrainerCallback):
    def __init__(self):
        self.training_loss = []
        self.steps = []
        self.epoch_loss = []
        self.epochs = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            self.training_loss.append(logs['loss'])
            self.steps.append(state.global_step)

    def on_epoch_end(self, args, state, control, **kwargs):
        if self.training_loss:
            avg_loss = sum(self.training_loss[-10:]) / min(10, len(self.training_loss))
            self.epoch_loss.append(avg_loss)
            self.epochs.append(state.epoch)


callback = MetricsCallback()
trainer.add_callback(callback)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=5e-5,
        num_train_epochs=5,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        optim="adamw_8bit",
        gradient_checkpointing=True,
        output_dir="./recipe_model",
        logging_steps=5,
        save_strategy="epoch",
        max_seq_length=768,
        dataset_text_field="text",
        seed=42,
        report_to="none",
    ),
)

# Add callback
trainer.add_callback(callback)

print("Training started...")
trainer.train()
trainer.save_model("./final_recipe_model")
tokenizer.save_pretrained("./final_recipe_model")

# Plot training loss curve
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(callback.steps, callback.training_loss, 'b-', alpha=0.6, label='Training Loss')
plt.xlabel('Training Steps')
plt.ylabel('Loss')
plt.title('Training Loss Curve')
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(callback.epochs, callback.epoch_loss, 'r-o', label='Epoch Loss')
plt.xlabel('Epoch')
plt.ylabel('Average Loss')
plt.title('Loss per Epoch')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.savefig('training_loss_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Training completed!")
print(f"Final loss: {callback.training_loss[-1]:.4f}")
print(f"Total steps: {callback.steps[-1]}")
print(f"Loss curves saved as 'training_loss_curves.png'")

# ================================
# CELL 5: Load Trained Model
# ================================

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./final_recipe_model",
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

def generate_recipe(dish_name, about="", ingredients=""):
    instruction = f"Generate a complete recipe for {dish_name}."
    if about:
        instruction += f"\nAbout: {about}"
    if ingredients:
        instruction += f"\nIngredients: {ingredients}"

    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = tokenizer([prompt], return_tensors="pt", truncation=True, max_length=400)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.6,
            top_p=0.85,
            top_k=25,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    recipe = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    # Clean output
    sentences = [s.strip() for s in recipe.split('.') if s.strip()]
    unique_sentences = []
    for sentence in sentences:
        if sentence not in unique_sentences and len(sentence) > 5:
            unique_sentences.append(sentence)
        if len(unique_sentences) >= 6:
            break

    return '. '.join(unique_sentences) + '.'

# Test generation
test_recipe = generate_recipe("Jollof Rice", "West African rice dish", "Rice, tomatoes, spices")
print("Generated Recipe:")
print("=" * 40)
print(test_recipe)

# ================================
# CELL 6: Interactive Generator
# ================================

import ipywidgets as widgets
from IPython.display import display, clear_output

dish_input = widgets.Text(value='Egusi Soup', description='Dish:')
about_input = widgets.Textarea(value='Nigerian soup', description='About:')
ingredients_input = widgets.Textarea(value='Melon seeds, spinach', description='Ingredients:')
generate_button = widgets.Button(description='Generate Recipe', button_style='success')
output = widgets.Output()

def on_click(b):
    with output:
        clear_output(wait=True)
        recipe = generate_recipe(dish_input.value, about_input.value, ingredients_input.value)
        print(f"Recipe for {dish_input.value}:")
        print("=" * 40)
        print(recipe)

generate_button.on_click(on_click)
display(widgets.VBox([dish_input, about_input, ingredients_input, generate_button, output]))

# ================================
# CELL 7: Complete Evaluation with Chapter 4 Visualizations
# ================================

import numpy as np
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
import matplotlib.pyplot as plt
import seaborn as sns
import time

class Chapter4Evaluator:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.rouge_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        self.smoothing = SmoothingFunction()

    def run_complete_evaluation(self):
        print("Running Complete Chapter 4 Evaluation...")
        print("=" * 60)

        # Generate test recipes
        test_dishes = [
            {"name": "Jollof Rice", "about": "West African rice", "ingredients": "Rice, tomatoes"},
            {"name": "Egusi Soup", "about": "Nigerian soup", "ingredients": "Melon seeds, spinach"},
            {"name": "Moi-Moi", "about": "Bean pudding", "ingredients": "Black-eyed peas"},
            {"name": "Ugali", "about": "East African staple", "ingredients": "Cornmeal, water"},
            {"name": "Suya", "about": "Grilled meat", "ingredients": "Beef, spices"},
            {"name": "Fufu", "about": "Starchy side dish", "ingredients": "Cassava"},
            {"name": "Injera", "about": "Ethiopian bread", "ingredients": "Teff flour"},
            {"name": "Pounded Yam", "about": "Nigerian staple", "ingredients": "Yam"}
        ]

        predictions = []
        references = []
        generation_times = []

        for dish in test_dishes:
            start_time = time.time()
            pred = generate_recipe(dish['name'], dish['about'], dish['ingredients'])
            gen_time = time.time() - start_time

            predictions.append(pred)
            generation_times.append(gen_time)

            ref = f"{dish['name']} is {dish['about']} made with {dish['ingredients']}. Prepare ingredients. Cook according to traditional method. Serve hot."
            references.append(ref)

        # Calculate all metrics
        results = self.calculate_metrics(predictions, references, generation_times)

        # Display results
        self.display_results(results)

        # Generate visualizations
        self.create_visualizations(results, predictions)

        return results

    def calculate_metrics(self, predictions, references, generation_times):
        results = {}

        # BLEU scores
        bleu_scores = []
        for pred, ref in zip(predictions, references):
            pred_tokens = word_tokenize(pred.lower())
            ref_tokens = word_tokenize(ref.lower())
            score = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=self.smoothing.method1)
            bleu_scores.append(score)

        results['bleu_scores'] = bleu_scores
        results['bleu_mean'] = np.mean(bleu_scores)
        results['bleu_std'] = np.std(bleu_scores)

        # ROUGE scores
        rouge1_scores, rouge2_scores, rougeL_scores = [], [], []
        for pred, ref in zip(predictions, references):
            scores = self.rouge_scorer.score(ref, pred)
            rouge1_scores.append(scores['rouge1'].fmeasure)
            rouge2_scores.append(scores['rouge2'].fmeasure)
            rougeL_scores.append(scores['rougeL'].fmeasure)

        results['rouge1_scores'] = rouge1_scores
        results['rouge1_mean'] = np.mean(rouge1_scores)
        results['rouge1_std'] = np.std(rouge1_scores)

        results['rouge2_scores'] = rouge2_scores
        results['rouge2_mean'] = np.mean(rouge2_scores)
        results['rouge2_std'] = np.std(rouge2_scores)

        results['rougeL_scores'] = rougeL_scores
        results['rougeL_mean'] = np.mean(rougeL_scores)
        results['rougeL_std'] = np.std(rougeL_scores)

        # METEOR scores
        meteor_scores = []
        for pred, ref in zip(predictions, references):
            pred_tokens = word_tokenize(pred.lower())
            ref_tokens = word_tokenize(ref.lower())
            score = meteor_score([ref_tokens], pred_tokens)
            meteor_scores.append(score)

        results['meteor_scores'] = meteor_scores
        results['meteor_mean'] = np.mean(meteor_scores)
        results['meteor_std'] = np.std(meteor_scores)

        # Perplexity
        perplexities = []
        for text in predictions:
            inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = self.model(**inputs, labels=inputs["input_ids"])
                perplexity = torch.exp(outputs.loss).item()
                perplexities.append(min(perplexity, 1000))

        results['perplexity_scores'] = perplexities
        results['perplexity_mean'] = np.mean(perplexities)
        results['perplexity_std'] = np.std(perplexities)

        # Model efficiency metrics
        total_params = sum(p.numel() for p in self.model.parameters())
        trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)

        results['total_parameters'] = total_params
        results['trainable_parameters'] = trainable_params
        results['parameter_efficiency'] = (trainable_params / total_params) * 100

        # Generation metrics
        results['generation_times'] = generation_times
        results['avg_generation_time'] = np.mean(generation_times)
        results['std_generation_time'] = np.std(generation_times)

        # Memory usage
        if torch.cuda.is_available():
            results['memory_allocated_gb'] = torch.cuda.memory_allocated() / 1024**3
            results['memory_reserved_gb'] = torch.cuda.memory_reserved() / 1024**3
        else:
            results['memory_allocated_gb'] = 0
            results['memory_reserved_gb'] = 0

        # Recipe quality metrics
        avg_length = np.mean([len(pred.split()) for pred in predictions])
        results['avg_recipe_length'] = avg_length

        return results

    def display_results(self, results):
        print("\nCHAPTER 4: EVALUATION RESULTS")
        print("=" * 60)

        print("\nTABLE 4.1: AUTOMATIC EVALUATION METRICS")
        print("-" * 60)
        print(f"{'Metric':<20} {'Mean':<12} {'Std Dev':<12}")
        print("-" * 60)
        print(f"{'BLEU Score':<20} {results['bleu_mean']:<12.4f} {results['bleu_std']:<12.4f}")
        print(f"{'ROUGE-1':<20} {results['rouge1_mean']:<12.4f} {results['rouge1_std']:<12.4f}")
        print(f"{'ROUGE-2':<20} {results['rouge2_mean']:<12.4f} {results['rouge2_std']:<12.4f}")
        print(f"{'ROUGE-L':<20} {results['rougeL_mean']:<12.4f} {results['rougeL_std']:<12.4f}")
        print(f"{'METEOR':<20} {results['meteor_mean']:<12.4f} {results['meteor_std']:<12.4f}")
        print(f"{'Perplexity':<20} {results['perplexity_mean']:<12.2f} {results['perplexity_std']:<12.2f}")

        print("\n\nTABLE 4.2: TRAINING EFFICIENCY METRICS")
        print("-" * 60)
        print(f"{'Metric':<30} {'Value':<20}")
        print("-" * 60)
        print(f"{'Total Parameters':<30} {results['total_parameters']:,}")
        print(f"{'Trainable Parameters':<30} {results['trainable_parameters']:,}")
        print(f"{'Parameter Efficiency (%)':<30} {results['parameter_efficiency']:.2f}%")
        print(f"{'Memory Allocated (GB)':<30} {results['memory_allocated_gb']:.2f}")
        print(f"{'Avg Generation Time (s)':<30} {results['avg_generation_time']:.3f}")
        print(f"{'Avg Recipe Length (words)':<30} {results['avg_recipe_length']:.1f}")

        print("\n" + "=" * 60)

    def create_visualizations(self, results, predictions):
        """Create Chapter 4 visualizations"""

        # Set style
        sns.set_style("whitegrid")

        # Create figure with multiple subplots
        fig = plt.figure(figsize=(16, 12))

        # 1. Metrics Comparison Bar Chart
        ax1 = plt.subplot(3, 3, 1)
        metrics = ['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'METEOR']
        values = [results['bleu_mean'], results['rouge1_mean'],
                 results['rouge2_mean'], results['rougeL_mean'], results['meteor_mean']]
        colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
        ax1.bar(metrics, values, color=colors, alpha=0.7)
        ax1.set_ylabel('Score')
        ax1.set_title('Figure 4.1: Automatic Metrics Comparison')
        ax1.set_ylim(0, 1)
        for i, v in enumerate(values):
            ax1.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)

        # 2. Metrics Distribution Box Plot
        ax2 = plt.subplot(3, 3, 2)
        metric_data = [results['bleu_scores'], results['rouge1_scores'],
                       results['rouge2_scores'], results['rougeL_scores'],
                       results['meteor_scores']]
        bp = ax2.boxplot(metric_data, labels=['BLEU', 'R-1', 'R-2', 'R-L', 'MET'], patch_artist=True)
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
        ax2.set_ylabel('Score')
        ax2.set_title('Figure 4.2: Metrics Distribution')
        ax2.grid(True, alpha=0.3)

        # 3. Parameter Efficiency Pie Chart
        ax3 = plt.subplot(3, 3, 3)
        trainable = results['trainable_parameters']
        frozen = results['total_parameters'] - trainable
        ax3.pie([trainable, frozen], labels=['Trainable', 'Frozen'],
               colors=['#2ecc71', '#95a5a6'], autopct='%1.1f%%', startangle=90)
        ax3.set_title('Figure 4.3: Parameter Efficiency')

        # 4. Generation Time Analysis
        ax4 = plt.subplot(3, 3, 4)
        ax4.plot(results['generation_times'], 'o-', color='#3498db', linewidth=2, markersize=8)
        ax4.axhline(y=results['avg_generation_time'], color='r', linestyle='--',
                   label=f'Mean: {results["avg_generation_time"]:.3f}s')
        ax4.set_xlabel('Recipe Sample')
        ax4.set_ylabel('Time (seconds)')
        ax4.set_title('Figure 4.4: Generation Time per Recipe')
        ax4.legend()
        ax4.grid(True, alpha=0.3)

        # 5. Perplexity Distribution
        ax5 = plt.subplot(3, 3, 5)
        ax5.hist(results['perplexity_scores'], bins=8, color='#e74c3c', alpha=0.7, edgecolor='black')
        ax5.axvline(x=results['perplexity_mean'], color='blue', linestyle='--',
                   linewidth=2, label=f'Mean: {results["perplexity_mean"]:.2f}')
        ax5.set_xlabel('Perplexity')
        ax5.set_ylabel('Frequency')
        ax5.set_title('Figure 4.5: Perplexity Distribution')
        ax5.legend()

        # 6. Recipe Length Distribution
        ax6 = plt.subplot(3, 3, 6)
        recipe_lengths = [len(pred.split()) for pred in predictions]
        ax6.hist(recipe_lengths, bins=8, color='#9b59b6', alpha=0.7, edgecolor='black')
        ax6.axvline(x=np.mean(recipe_lengths), color='red', linestyle='--',
                   linewidth=2, label=f'Mean: {np.mean(recipe_lengths):.1f}')
        ax6.set_xlabel('Words')
        ax6.set_ylabel('Frequency')
        ax6.set_title('Figure 4.6: Generated Recipe Length')
        ax6.legend()

        # 7. Metrics Heatmap
        ax7 = plt.subplot(3, 3, 7)
        metrics_matrix = np.array([results['bleu_scores'], results['rouge1_scores'],
                                   results['rouge2_scores'], results['rougeL_scores'],
                                   results['meteor_scores']])
        im = ax7.imshow(metrics_matrix, cmap='YlOrRd', aspect='auto')
        ax7.set_yticks(range(5))
        ax7.set_yticklabels(['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'METEOR'])
        ax7.set_xticks(range(len(predictions)))
        ax7.set_xticklabels([f'R{i+1}' for i in range(len(predictions))], fontsize=8)
        ax7.set_title('Figure 4.7: Metrics Heatmap per Recipe')
        plt.colorbar(im, ax=ax7)

        # 8. Memory Usage
        ax8 = plt.subplot(3, 3, 8)
        memory_data = [results['memory_allocated_gb'], results['memory_reserved_gb']]
        ax8.bar(['Allocated', 'Reserved'], memory_data, color=['#3498db', '#95a5a6'], alpha=0.7)
        ax8.set_ylabel('Memory (GB)')
        ax8.set_title('Figure 4.8: GPU Memory Usage')
        for i, v in enumerate(memory_data):
            ax8.text(i, v + 0.05, f'{v:.2f} GB', ha='center')

        # 9. Model Statistics Summary
        ax9 = plt.subplot(3, 3, 9)
        ax9.axis('off')
        summary_text = f"""
        Model Performance Summary

        Best Metric: ROUGE-L ({results['rougeL_mean']:.4f})
        Lowest Perplexity: {min(results['perplexity_scores']):.2f}
        Fastest Generation: {min(results['generation_times']):.3f}s

        Efficiency:
        - Trainable Params: {results['parameter_efficiency']:.2f}%
        - Avg Generation: {results['avg_generation_time']:.3f}s
        - Memory Usage: {results['memory_allocated_gb']:.2f} GB
        """
        ax9.text(0.1, 0.5, summary_text, fontsize=10, family='monospace',
                verticalalignment='center')
        ax9.set_title('Table 4.3: Performance Summary')

        plt.tight_layout()
        plt.savefig('chapter4_evaluation_results.png', dpi=300, bbox_inches='tight')
        plt.show()

        print("\nVisualization saved as 'chapter4_evaluation_results.png'")

        # Save individual figures for thesis
        self.save_individual_figures(results, predictions)

    def save_individual_figures(self, results, predictions):
        """Save individual figures for thesis"""

        # Figure 4.1: Metrics Bar Chart
        plt.figure(figsize=(8, 5))
        metrics = ['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'METEOR']
        values = [results['bleu_mean'], results['rouge1_mean'],
                 results['rouge2_mean'], results['rougeL_mean'], results['meteor_mean']]
        colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
        plt.bar(metrics, values, color=colors, alpha=0.7)
        plt.ylabel('Score', fontsize=12)
        plt.title('Automatic Metrics Comparison', fontsize=14, fontweight='bold')
        plt.ylim(0, 1)
        for i, v in enumerate(values):
            plt.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig('figure_4_1_metrics_comparison.png', dpi=300, bbox_inches='tight')
        plt.close()

        print("Individual figures saved for thesis")

# Run complete evaluation
evaluator = Chapter4Evaluator(model, tokenizer)
evaluation_results = evaluator.run_complete_evaluation()

print("\nChapter 4 Results Package Complete!")
print("Files generated:")
print("- training_loss_curves.png")
print("- chapter4_evaluation_results.png")
print("- figure_4_1_metrics_comparison.png")
print("\nAll results ready for your thesis!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Saving Recipe.csv to Recipe (1).csv
Processed 4722 recipes
==((====))==  Unsloth 2025.9.9: Fast Gemma2 patching. Transformers: 4.56.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
trainable params: 10,383,360 || all params: 2,624,725,248 || trainable%: 0.3956


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/4722 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Training started...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,722 | Num Epochs = 5 | Total steps = 2,955
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 10,383,360 of 2,624,725,248 (0.40% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,2.382900
10,2.324000
15,2.353900
20,2.205300
25,2.311800
30,2.317900
35,2.400400
40,2.166100
45,2.229400
50,2.313400


Step,Training Loss
5,2.382900
10,2.324000
15,2.353900
20,2.205300
25,2.311800
30,2.317900
35,2.400400
40,2.166100
45,2.229400
50,2.313400
